In [ ]:
import pandas as pd

# Load the dataset
csv_path = "Detailed_Statistics_Departures.csv"
df = pd.read_csv(csv_path, low_memory=False)

# Display the shape and the first few rows of the DataFrame
print("Loaded df:", df.shape)
df.head()

Loaded df: (1048572, 17)


,Carrier Code,Date (MM/DD/YYYY),Flight Number,Tail Number,Destination Airport,Scheduled departure time,Actual departure time,Scheduled elapsed time (Minutes),Actual elapsed time (Minutes),Departure delay (Minutes),Wheels-off time,Taxi-Out time (Minutes),Delay Carrier (Minutes),Delay Weather (Minutes),Delay National Aviation System (Minutes),Delay Security (Minutes),Delay Late Aircraft Arrival (Minutes)
0,AS,1/1/2016,15,N552AS,SEA,18:45,19:10,400,373,25,19:26,16,0,0,0,0,0
1,AS,1/1/2016,25,N492AS,SEA,7:00,7:03,405,371,3,7:25,22,0,0,0,0,0
2,AS,1/1/2016,33,N549AS,PDX,17:10,17:20,399,332,10,17:32,12,0,0,0,0,0
3,AS,1/1/2016,769,N526AS,SAN,18:15,18:12,419,370,-3,18:31,19,0,0,0,0,0
4,AS,1/1/2017,15,N568AS,SEA,18:50,21:40,400,384,170,22:08,28,36,0,0,0,118


In [ ]:
#Featue engineering: Create new features from the Date and Scheduled departure time columns

import numpy as np
import pandas as pd

# Convert Date
df["Date (MM/DD/YYYY)"] = pd.to_datetime(df["Date (MM/DD/YYYY)"], errors="coerce")

# Extract Month and Day of Week
df["Month"] = df["Date (MM/DD/YYYY)"].dt.month
df["Day_of_Week"] = df["Date (MM/DD/YYYY)"].dt.dayofweek  # Monday=0 ... Sunday=6

# Convert Scheduled departure time (HH:MM) -> hour
# IMPORTANT: Don't overwrite original if you want to keep it; we can parse a copy
sched_time = df["Scheduled departure time"].astype(str).str.strip()
sched_dt = pd.to_datetime(sched_time, format="%H:%M", errors="coerce")
df["Scheduled_Hour"] = sched_dt.dt.hour

# Season mapping
def get_season(m):
    if m in [3, 4, 5]:
        return "Spring"
    elif m in [6, 7, 8]:
        return "Summer"
    elif m in [9, 10, 11]:
        return "Fall"
    else:
        return "Winter"

df["Season"] = df["Month"].apply(get_season)

print("Done: engineered Month, Day_of_Week, Scheduled_Hour, Season")
df[["Date (MM/DD/YYYY)", "Month", "Day_of_Week", "Scheduled departure time", "Scheduled_Hour", "Season"]].head(10)

Done: engineered Month, Day_of_Week, Scheduled_Hour, Season


,Date (MM/DD/YYYY),Month,Day_of_Week,Scheduled departure time,Scheduled_Hour,Season
0,2016-01-01,1,4,18:45,18,Winter
1,2016-01-01,1,4,7:00,7,Winter
2,2016-01-01,1,4,17:10,17,Winter
3,2016-01-01,1,4,18:15,18,Winter
4,2017-01-01,1,6,18:50,18,Winter
5,2017-01-01,1,6,7:00,7,Winter
6,2017-01-01,1,6,16:35,16,Winter
7,2017-01-01,1,6,18:10,18,Winter
8,2018-01-01,1,0,18:59,18,Winter
9,2018-01-01,1,0,16:23,16,Winter


In [6]:
# Target variable: Create a binary target for whether the departure delay is greater than 20 minutes

TARGET_COL = "Departure delay (Minutes)"
DELAY_THRESHOLD_MIN = 20

# Drop rows with missing target
df_model = df[df[TARGET_COL].notna()].copy()

# Binary target
df_model["Delayed"] = (df_model[TARGET_COL] > DELAY_THRESHOLD_MIN).astype(int)

print("Delay rate:", df_model["Delayed"].mean().round(4))
df_model[["Departure delay (Minutes)", "Delayed"]].head()

Delay rate: 0.1701


,Departure delay (Minutes),Delayed
0,25,1
1,3,0
2,10,0
3,-3,0
4,170,1


In [7]:
# Define features and target for modeling

FEATURES = [
    "Carrier Code",
    "Destination Airport",
    "Tail Number",        # Option B (you said OK)
    "Scheduled_Hour",
    "Month",
    "Season",
    "Day_of_Week"
]

X = df_model[FEATURES].copy()
y = df_model["Delayed"].copy()

print("Features used:", FEATURES)
print("X shape:", X.shape, "| y shape:", y.shape)
X.head()

Features used: ['Carrier Code', 'Destination Airport', 'Tail Number', 'Scheduled_Hour', 'Month', 'Season', 'Day_of_Week']
X shape: (1048572, 7) | y shape: (1048572,)


,Carrier Code,Destination Airport,Tail Number,Scheduled_Hour,Month,Season,Day_of_Week
0,AS,SEA,N552AS,18,1,Winter,4
1,AS,SEA,N492AS,7,1,Winter,4
2,AS,PDX,N549AS,17,1,Winter,4
3,AS,SAN,N526AS,18,1,Winter,4
4,AS,SEA,N568AS,18,1,Winter,6


In [11]:
# ============================================================
# Modeling (Random Forest with proper encoding)
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

# 1) split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=2026,
    stratify=y
)

# 2) identify categorical vs numeric columns
cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
num_cols = X_train.select_dtypes(include=["number"]).columns.tolist()

print("Categorical columns:", cat_cols)
print("Numeric columns:", num_cols)

# 3) preprocessing
categorical_preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

numeric_preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("cat", categorical_preprocess, cat_cols),
        ("num", numeric_preprocess, num_cols)
    ]
)

# 4) model
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=2026,
    n_jobs=-1,
    class_weight="balanced_subsample"
)

# 5) pipeline: preprocess -> model
rf_pipeline = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", rf_model)
])

# 6) train
rf_pipeline.fit(X_train, y_train)

print("Model training completed (Random Forest + OneHot encoding).")

C:\Users\jxzor\AppData\Local\Temp\ipykernel_20112\2842833994.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()


Categorical columns: ['Carrier Code', 'Destination Airport', 'Tail Number', 'Season']
Numeric columns: ['Scheduled_Hour', 'Month', 'Day_of_Week']


KeyboardInterrupt: 